In [ ]:
# =================================================================
# 1. CONFIGURATION (MULTI-SUBJECT DYNAMIC)
# =================================================================
import step1_loading
import step2_preprocessing
import step3_epoching
import step4_pseudotrials
import step5_decoding
import step6_visualization
import step7_comparison
import numpy as np
import pandas as pd

# Dynamically generate Subject IDs from 01 to 34
START_SUB = 1
END_SUB = 1
SUBJECT_IDS = [f"sub-{i:02d}" for i in range(START_SUB, END_SUB + 1)]

BASIC_PATH = 'project/ds006761'

print(f"--- CONFIGURATION ---")
print(f"Target Subjects: {len(SUBJECT_IDS)} subjects")
print(f"Range: {SUBJECT_IDS[0]} to {SUBJECT_IDS[-1]}")

# Storage for Grand Average Analysis
# 1. Main Decoding
group_results = {1: [], 2: []} 

# 2. Comparative Decoding (Grouping by Game Outcome)
keys = ['Own Current', 'Opponent Current', 'Own Previous', 'Opponent Previous']
group_comp_results = {
    'Winner': {k: [] for k in keys},
    'Loser':  {k: [] for k in keys}
}

print(f"\n--- STARTING PIPELINE ---")

# =================================================================
# 2. OUTER LOOP: SUBJECTS
# =================================================================
for subject_id in SUBJECT_IDS:
    print(f"\n" + "#"*60)
    print(f"   PROCESSING SUBJECT: {subject_id}")
    print("#"*60)
    
    # Load Data for this Subject
    try:
        raw_p1_full, raw_p2_full = step1_loading.load_and_split_data(subject_id, BASIC_PATH)
    except Exception as e:
        print(f"❌ Failed to load {subject_id}: {e}")
        continue


    # --- DETERMINE GAME WINNER (Logic added) ---
    # We read the events file to count wins
    try:
        ev_file = f'{BASIC_PATH}/{subject_id}/eeg/{subject_id}_task-RPS_events.tsv'
        ev_df = pd.read_csv(ev_file, sep='\t')
        
        # Calculate scores (1=Rock, 2=Paper, 3=Scissors)
        # Logic: (P1 - P2) % 3 == 1 means P1 wins.
        p1_vals = ev_df['player1_resp'].values
        p2_vals = ev_df['player2_resp'].values
        
        # Count wins excluding NaNs
        valid_mask = np.isfinite(p1_vals) & np.isfinite(p2_vals)
        diff = (p1_vals[valid_mask] - p2_vals[valid_mask]) % 3
        
        p1_wins = np.sum(diff == 1)
        p2_wins = np.sum(diff == 2)
        
        subject_winner = 1 if p1_wins >= p2_wins else 2
        print(f"   -> Game Outcome: P1({p1_wins}) vs P2({p2_wins}) => Winner is Player {subject_winner}")
        
    except Exception as e:
        print(f"   ⚠️ Could not determine winner, defaulting to P1: {e}")
        subject_winner = 1
    # -------------------------------------------

# =================================================================
# 3. INNER LOOP: PLAYERS (1 & 2)
# =================================================================
for player_num, raw_data in zip([1, 2], [raw_p1_full, raw_p2_full]):
    
    if raw_data is None:
        print(f"   -> Skipping Player {player_num} (No Data)")
        continue
        
    print(f"\n   --- Player {player_num} Analysis ---")

    # A. PREPROCESSING + ICA
    raw_clean = step2_preprocessing.run_preprocessing(raw_data, subject_id, BASIC_PATH, player_num)

    # B. EPOCHING
    epochs_tuple, full_df = step3_epoching.run_epoching(raw_clean, subject_id, BASIC_PATH, player_num)

    # C. TIME BINNING
    epochs_binned = step4_pseudotrials.bin_and_stitch_time_course(epochs_tuple)

    labels = full_df[f'player{player_num}_resp'].values
    X_pseudo, y_pseudo = step4_pseudotrials.create_pseudo_trials_by_averaging(
        epochs_binned, labels, 
        random_seed=1  # Paper uses 1, not 42
    )

    # D. MAIN DECODING (Own Current)
    times, mean_scores, std_scores, fold_accuracies = step5_decoding.run_svm_decoding_with_pseudotrials(
        X_pseudo, y_pseudo
    )
    
    if mean_scores is not None:
        # Store all statistics
        group_results[player_num].append({
            'mean_scores': mean_scores,
            'std_scores': std_scores,
            'fold_accuracies': fold_accuracies,
            'subject_id': subject_id,
            'player_num': player_num
        })
        print(f"   -> ✅ Main Scores stored (mean: {np.mean(mean_scores):.2f}%)")
    else:
        print("   -> ❌ Main Decoding failed.")

    # Step G: Comparative Analysis
    comp_results = step7_comparison.run_comparative_analysis(epochs_binned, full_df, player_num)

    # Store these scores based on Winner/Loser status
    if player_num == subject_winner:
        group_key = 'Winner'
    else:
        group_key = 'Loser'

    for task_name, task_scores in comp_results.items():
        if task_scores is not None:
            # Append to 'Winner' or 'Loser' dict
            group_comp_results[group_key][task_name].append(task_scores)
    
    print(f"   -> ✅ Comparative Scores stored in '{group_key}' group.")

# =================================================================
# 4. FINAL STEP: GRAND AVERAGE VISUALIZATION WITH STATISTICS
# =================================================================
print("\n" + "="*60)
print("   GENERATING GRAND AVERAGE PLOTS WITH STATISTICS")
print("="*60)

# Plot individual player results with statistics
for player_num in [1, 2]:
    if group_results[player_num]:
        print(f"\n--- Player {player_num} Statistics ---")
        
        # Get the first subject for example plots
        first_result = group_results[player_num][0]
        
        # Plot with confidence intervals for one subject
        time_axis = np.linspace(0, 5.0, len(first_result['mean_scores']))
        step6_visualization.plot_decoding_with_confidence(
            time_axis, 
            first_result['mean_scores'], 
            first_result['std_scores'],
            title=f"Player {player_num} - {first_result['subject_id']}"
        )
        
        # Plot fold variability for one subject
        step6_visualization.plot_fold_variability(
            first_result['fold_accuracies'], 
            time_axis,
            title=f"Fold Variability - Player {player_num}"
        )
        
        # Plot grand average across all subjects
        step6_visualization.plot_grand_average_with_stats(group_results, player_num)

# Plot comparative results (Winners vs Losers)
grand_time_axis = np.linspace(0, 5.0, 20)

for group_name in ['Winner', 'Loser']:
    print(f"\n--- {group_name} Grand Averages ---")
    
    # Check if we have data
    if len(group_comp_results[group_name]['Own Current']) > 0:
        # Plot Comparative Decoding (4 Lines)
        step7_comparison.plot_grand_average_comparison(
            group_comp_results[group_name], 
            group_name
        )
        
        # Calculate and print statistics for winners/losers
        all_scores = group_comp_results[group_name]['Own Current']
        mean_scores = np.mean(all_scores, axis=0)
        std_scores = np.std(all_scores, axis=0)
        
        print(f"  {group_name} Statistics:")
        print(f"    N: {len(all_scores)}")
        print(f"    Mean accuracy: {np.mean(mean_scores):.2f}%")
        print(f"    Peak accuracy: {np.max(mean_scores):.2f}% at bin {np.argmax(mean_scores)}")
    else:
        print(f"  No data for {group_name} group")

print("\n✅ Multi-Subject Analysis Complete with Enhanced Statistics!")

# =================================================================
# 5. COMPREHENSIVE STATISTICS SUMMARY
# =================================================================
print("\n" + "="*60)
print("   COMPREHENSIVE STATISTICS SUMMARY")
print("="*60)

# Calculate overall statistics
for player_num in [1, 2]:
    if group_results[player_num]:
        # Extract all mean scores
        all_means = [result['mean_scores'] for result in group_results[player_num]]
        mean_matrix = np.array(all_means)
        
        # Overall statistics
        grand_mean = np.mean(mean_matrix, axis=0)
        overall_mean = np.mean(grand_mean)
        overall_std = np.std(mean_matrix.flatten())
        
        print(f"\nPlayer {player_num} (N={len(group_results[player_num])}):")
        print(f"  Overall accuracy: {overall_mean:.2f}% ± {overall_std:.2f}%")
        
        # Check against chance (33.33%)
        from scipy import stats
        t_stat, p_value = stats.ttest_1samp(mean_matrix.flatten(), 33.33)
        significance = "SIGNIFICANT" if p_value < 0.05 else "NOT SIGNIFICANT"
        print(f"  Against chance (33.33%): t={t_stat:.3f}, p={p_value:.4f} ({significance})")

# Winner vs Loser comparison
if len(group_comp_results['Winner']['Own Current']) > 0 and len(group_comp_results['Loser']['Own Current']) > 0:
    print(f"\nWinner vs Loser Comparison:")
    
    winner_scores = np.mean(group_comp_results['Winner']['Own Current'], axis=1)
    loser_scores = np.mean(group_comp_results['Loser']['Own Current'], axis=1)
    
    print(f"  Winners (N={len(winner_scores)}): {np.mean(winner_scores):.2f}% ± {np.std(winner_scores):.2f}%")
    print(f"  Losers (N={len(loser_scores)}): {np.mean(loser_scores):.2f}% ± {np.std(loser_scores):.2f}%")
    
    # T-test between groups
    from scipy import stats
    t_stat, p_value = stats.ttest_ind(winner_scores, loser_scores, equal_var=False)
    significance = "SIGNIFICANT" if p_value < 0.05 else "NOT SIGNIFICANT"
    print(f"  Group difference: t={t_stat:.3f}, p={p_value:.4f} ({significance})")